In [86]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from diffusers import AutoencoderDC
import numpy as np
import math

In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [88]:
class ConcatenateImgTextMLP(nn.Module):
    def __init__(self, inputDimension, outputDimension, isSame = False):
        super().__init__()
        if isSame:
            self.layer1 = nn.Linear(inputDimension, inputDimension)
            self.layer2 = nn.Linear(inputDimension, outputDimension)
        else:
            self.layer1 = nn.Linear(inputDimension, inputDimension//2)
            self.layer2 = nn.Linear(inputDimension//2, outputDimension)
        
        self.gelu = nn.GELU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.gelu(x)
        x = self.layer2(x)
        return x


txt_input = torch.randn(32, 512, 5632)
img_input = torch.randn(32, 64, 768)
concatInpstxt = ConcatenateImgTextMLP(5632, 768)
concatInpsimg = ConcatenateImgTextMLP(768, 768, isSame = True)

txt_input = concatInpstxt(txt_input)
img_input = concatInpsimg(img_input)

modalityEmbeds = nn.Embedding(2, 768)

print(txt_input.shape, img_input.shape)

img_embed = img_input + modalityEmbeds(torch.zeros(64, dtype=torch.long))
txt_embed = txt_input + modalityEmbeds(torch.ones(512, dtype=torch.long))


print(txt_embed.shape, img_embed.shape)
out = torch.concat([img_embed, txt_embed], dim=1)
out.shape

torch.Size([32, 512, 768]) torch.Size([32, 64, 768])
torch.Size([32, 512, 768]) torch.Size([32, 64, 768])


torch.Size([32, 576, 768])

In [ ]:
x = torch.randn(2, 3, 512, 512)
dc_ae = AutoencoderDC.from_pretrained("mit-han-lab/dc-ae-f64c128-in-1.0-diffusers", torch_dtype=torch.float32)
with torch.no_grad():
    latents = dc_ae.encode(x).latent
latents.shape
flattened = latents.flatten(2)
flattened.shape

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, embedDimension):
        super().__init__()
        self.embedDimension = embedDimension
        self.linear1 = nn.Linear(embedDimension, 4 * embedDimension)
        self.silu = nn.SiLU()
        self.outlayer = nn.Linear(4 * embedDimension, embedDimension)

        nn.init.normal_(self.linear1.weight, std=0.02)
        nn.init.normal_(self.outlayer.weight, std=0.02)

    def forward(self, t):

        half = self.embedDimension// 2
        exponent = -math.log(10000) * torch.arange(0, half, dtype=torch.float32) / half
        freq = torch.exp(exponent.to(device))

        # timedimMap = t.float().unsqueeze(0) * freq[None, :]
        timedimMap = t[:, None].float() * freq[None, :]
        sinusoidal = torch.cat([torch.cos(timedimMap), torch.sin(timedimMap)], dim = -1)

        sinusoidal = self.linear1(sinusoidal)
        sinusoidal = self.silu(sinusoidal)
        out = self.outlayer(sinusoidal)

        return out

tEmbed = TimeEmbedding(768)
tEmbed.to(device)
time = torch.tensor([5]).to(device)
out = tEmbed(time)
out.shape

torch.Size([1, 768])

In [ ]:
class AdaptiveLayerNorm(nn.Module):
    def __init__(self, embedDimension):
        super().__init__()
        self.embedDimension = embedDimension
        self.adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(embedDimension, 6 * embedDimension)
        )
        self.scaleShiftParameters = nn.Parameter(torch.zeros(6, embedDimension))
        nn.init.zeros_(self.adaLN[1].weight)
        nn.init.zeros_(self.adaLN[1].bias)      
    
    def forward(self, t):
        batchSize, _ = t.shape
        t = self.adaLN(t)
        t = t.reshape(batchSize, 6, -1)
        gamma_msa, beta_msa, alpha_msa, gamma_mlp, beta_mlp, alpha_mlp = (
            (self.scaleShiftParameters[None] + t).chunk(6, dim = 1)
        )
        gamma_msa = gamma_msa.squeeze(1)
        beta_msa = beta_msa.squeeze(1)
        alpha_msa = alpha_msa.squeeze(1)
        gamma_mlp = gamma_mlp.squeeze(1)
        beta_mlp = beta_mlp.squeeze(1)
        alpha_mlp = alpha_mlp.squeeze(1)
        return gamma_msa, beta_msa, alpha_msa, gamma_mlp, beta_mlp, alpha_mlp
    
embedDimension = 768
tEmbed = TimeEmbedding(embedDimension=embedDimension)
tEmbed.to(device)
time = torch.tensor([1000]).to(device)
tout = tEmbed(time)
adaNorm = AdaptiveLayerNorm(768)
g1, b1, a1, g2, b2, a2 = adaNorm(tout)
g1.shape, b1.shape, a1.shape, g2.shape, b2.shape, a2.shape

(torch.Size([1, 768]),
 torch.Size([1, 768]),
 torch.Size([1, 768]),
 torch.Size([1, 768]),
 torch.Size([1, 768]),
 torch.Size([1, 768]))

In [ ]:
import torch
import torch.nn as nn

class Rotary2DPositionalEncoding(nn.Module):
    def __init__(self, height, width, embedDimension):
        super().__init__()
        self.height = height
        self.width = width
        self.embedDimension = embedDimension

        self.dimHalf = embedDimension // 2
        self.dimQuarter = embedDimension // 4
        inverseFrequency = 1.0 / (10000 ** (torch.arange(0, self.dimQuarter, dtype=torch.float32) / self.dimQuarter))

        heightPositions = torch.arange(height, dtype=torch.float32)
        widthPositions = torch.arange(width, dtype=torch.float32)

        sinusoidHeight = torch.einsum("i,j->ij", heightPositions, inverseFrequency)
        sinusoidWidth = torch.einsum("i,j->ij", widthPositions, inverseFrequency)

        self.register_buffer("sinHeight", sinusoidHeight.sin(), persistent=False)
        self.register_buffer("cosHeight", sinusoidHeight.cos(), persistent=False)
        self.register_buffer("sinWidth", sinusoidWidth.sin(), persistent=False)
        self.register_buffer("cosWidth", sinusoidWidth.cos(), persistent=False)

    def rotateEveryTwo(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)
    
    def applyRope(self, x, sinHeight, cosHeight, sinWidth, cosWidth):

        xHeight = x[..., :self.dimHalf]
        xWidth = x[..., self.dimHalf:]

        sinHeight = sinHeight[None, :, None, :].to(x.device)
        cosHeight = cosHeight[None, :, None, :].to(x.device)
        sinWidth = sinWidth[None, None, :, :].to(x.device)
        cosWidth = cosWidth[None, None, :, :].to(x.device)

        xHeightRotional = (xHeight[..., :self.dimQuarter] * cosHeight) + (self.rotateEveryTwo(xHeight[..., :self.dimQuarter]) * sinHeight)
        xWidthRotional = (xWidth[..., :self.dimQuarter] * cosWidth) + (self.rotateEveryTwo(xWidth[..., :self.dimQuarter]) * sinWidth)

        xHeightRotional = torch.cat([xHeightRotional, xHeight[..., self.dimQuarter:]], dim=-1)
        xWidthRotional = torch.cat([xWidthRotional, xWidth[..., self.dimQuarter:]], dim=-1)
        rotated = torch.cat([xHeightRotional, xWidthRotional], dim=-1)
        return rotated


    def forward(self, x):
        B, L, D = x.shape
        assert D == self.embedDimension
        assert L == self.height * self.width, f"Expected seq_len {self.height*self.width}, got {L}"

        x = x.view(B, self.height, self.width, D)
        x = self.applyRope(x, self.sinHeight, self.cosHeight, self.sinWidth, self.cosWidth)
        return x.view(B, L, D)

rope2D = Rotary2DPositionalEncoding(8, 8, 768)
imagePatches = torch.randn(2, 64, 768)

out = rope2D(imagePatches)
out.shape

torch.Size([2, 64, 768])

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, imageSize, patchSize, inChannels, embedDimension):
        super().__init__()
        self.patchSize = patchSize
        self.inChannels = inChannels
        self.embedDimension = embedDimension
        self.imageSize = imageSize

        self.patches = imageSize//patchSize * imageSize//patchSize

        self.encode = nn.Conv2d(in_channels = inChannels, out_channels = embedDimension, kernel_size = patchSize, stride = patchSize, bias = True)
        self.decode = nn.ConvTranspose2d(in_channels=embedDimension, out_channels=inChannels, kernel_size=patchSize, stride=patchSize, bias=True)
        self.positionalEmbedding = nn.Parameter(torch.zeros(1, self.patches, embedDimension))
        nn.init.trunc_normal_(self.positionalEmbedding, std=0.02)

    def unPatchify(self, x):
        batchSize, NPatches, EmbedDim = x.shape
        patchPerDim = self.imageSize // self.patchSize
        x = x.transpose(1, 2).reshape(batchSize, EmbedDim, patchPerDim, patchPerDim)
        out = self.decode(x)
        return out


    def forward(self, latentImage):

        allPatch = self.encode(latentImage)
        # print(allPatch.shape)
        flattened = allPatch.flatten(2).transpose(1, 2)
        # print(flattened.shape, self.positionalEmbedding.shape)
        out = flattened + self.positionalEmbedding
        return out
    
latent = torch.randn(128, 8, 8).unsqueeze(0)
pEmbed = PatchEmbedding(imageSize = 8, patchSize = 2, inChannels = 128, embedDimension = 768)
out = pEmbed(latent)
unpatched = pEmbed.unPatchify(out)
out.shape, unpatched.shape

(torch.Size([1, 16, 768]), torch.Size([1, 128, 8, 8]))

In [ ]:
def shiftModulate(x, scale, shift):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class ScaleShiftBlock(nn.Module):
    def __init__(self, embedDimension):
        super().__init__()
        self.embedDimension = embedDimension
        self.norm = nn.LayerNorm(embedDimension, elementwise_affine=False, eps=1e-6)
        # nn.init.ones_(self.norm.weight)
        # nn.init.zeros_(self.norm.bias)

    def forward(self, x, beta, gamma):
        B, N, W = x.shape
        x_norm = self.norm(x)
        out = shiftModulate(x_norm, gamma, beta)
        return out


def scaleModulate(x, scale):
    return x * (1 + scale.unsqueeze(1))

class ScaleBlock(nn.Module):
    def __init__(self, embedDimension):
        super().__init__()
        self.embedDimension = embedDimension
        self.norm = nn.LayerNorm(embedDimension, elementwise_affine=False, eps=1e-6)
        # nn.init.ones_(self.norm.weight)
        # nn.init.zeros_(self.norm.bias)

    def forward(self, x, alpha):
        B, N, W = x.shape
        x_norm = self.norm(x)
        out = scaleModulate(x_norm, alpha)
        return out
    
# patchify_latents = torch.randn(1, 16, 768)
# scShft = ScaleBlock(embedDimension)
# out = scShft(patchify_latents, a1)
# out.shape

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embedDimension, numHeads, dropout = 0.2):
        super().__init__()

        assert embedDimension%numHeads == 0, "Embedding Dimension is Not Divisible By NumHeads"
        self.embedDimension = embedDimension
        self.numHeads = numHeads
        self.headDim = embedDimension//numHeads

        self.queryKeyValue = nn.Linear(embedDimension, embedDimension * 3, bias=False)
        self.drop = nn.Dropout(dropout)
        self.scale = self.headDim ** -0.5 
        self.outProjection = nn.Linear(embedDimension, embedDimension)

        nn.init.zeros_(self.queryKeyValue.weight)
        nn.init.zeros_(self.outProjection.weight)

    def forward(self, x):
        BatchSize, N, EmbedDim = x.shape

        qkv = self.queryKeyValue(x)
        qkv = qkv.reshape(BatchSize, N, 3, self.numHeads, EmbedDim // self.numHeads)
        q, k, v = qkv.unbind(2)
        attentionScore = (q @ k.transpose(-2, -1)) * self.scale
        attn = attentionScore.softmax(dim=-1)
        out = attn @ v 
        out = out.transpose(1, 2).reshape(BatchSize, N, EmbedDim)
        out = self.outProjection(out)
        out = self.drop(out)
        return out

In [ ]:
class FeedForwardBlock(nn.Module):
    def __init__(self, embedDimension):
        super().__init__()

        self.linear1 = nn.Linear(embedDimension, embedDimension * 4)
        self.linear2 = nn.Linear(embedDimension * 4, embedDimension)
        self.gelu = nn.GELU()

        nn.init.zeros_(self.linear1.weight)
        nn.init.zeros_(self.linear2.weight)

    def forward(self, x):
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.linear2(x)
        return x
    
latents = torch.randn(1, 16, 768)
ff = FeedForwardBlock(768)
out = ff(latents)
out.shape

torch.Size([1, 16, 768])

In [ ]:
class OutputHead(nn.Module):
    def __init__(self, embedDimension, outputDimension):
        super().__init__()

        self.linear1 = nn.Linear(embedDimension, embedDimension * 2)
        self.linear2 = nn.Linear(embedDimension * 2, outputDimension)
        self.gelu = nn.GELU()

        nn.init.zeros_(self.linear1.weight)
        nn.init.zeros_(self.linear2.weight)

    def forward(self, x):
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.linear2(x)
        return x
    
latents = torch.randn(1, 16, 768)
ff = OutputHead(768, 128)
out = ff(latents)
out.shape

torch.Size([1, 16, 128])

In [ ]:
class OutputBlock(nn.Module):
    def __init__(self, embedDimension, latentSize, latentChannels, patchSize, modelName="mit-han-lab/dc-ae-f64c128-in-1.0-diffusers"):
        super().__init__()
        self.dc_ae = AutoencoderDC.from_pretrained(modelName, torch_dtype=torch.float32)
        self.patchEmbedding = PatchEmbedding(imageSize = latentSize, patchSize = patchSize, inChannels = latentChannels, embedDimension = embedDimension)

    def forward(self, x):
        unpatchifiedTokens = self.patchEmbedding.unPatchify(x)
        with torch.no_grad():
            predictedNoise = self.dc_ae.decode(unpatchifiedTokens).sample
        return predictedNoise
    
dec = OutputBlock(embedDimension=784, latentSize=8, latentChannels=128, patchSize=2)
noisyImage = torch.randn(1, 128, 8, 8)

latents = torch.randn(1, 16, 784)
predictedNoise = dec(latents)
predictedNoise.shape

torch.Size([1, 3, 512, 512])

In [ ]:
class TinyRecursiveBlock(nn.Module):

    def __init__(self, embedDimension, numHeads, dropout = 0.2):
        super().__init__()

        self.embedDimension = embedDimension
        self.numHeads = numHeads
        self.scaleShiftBlock = ScaleShiftBlock(embedDimension)
        self.scaleBlock = ScaleBlock(embedDimension)
        self.yProjection = nn.Linear(64, 768)
        self.yrevProjection = nn.Linear(768, 64)
        self.multiHeadAttention = MultiHeadSelfAttention(embedDimension, numHeads, dropout)
        self.pointwiseFeedForward = FeedForwardBlock(embedDimension)
        self.output = OutputBlock(embedDimension, 8, 128, 2)
        self.outputHead = OutputHead(64, 64)


    def forward_reasoning(self, x, y, z, sharedParameters):
        gamma1, beta1, alpha1, gamma2, beta2, alpha2 = sharedParameters

        yReshaped = self.yProjection(y)
        xLen, yLen, zLen = x.shape[1], y.shape[1], z.shape[1]
        concatenatedInput = torch.cat([x, yReshaped, z], dim=1)
        initial = concatenatedInput

        scaleShiftOut1 = self.scaleShiftBlock(concatenatedInput, gamma1, beta1)
        selfAttentionOut1 = self.multiHeadAttention(scaleShiftOut1)
        scaleOut1 = self.scaleBlock(selfAttentionOut1, alpha1)

        initial =  initial + scaleOut1

        scaleShiftOut2 = self.scaleShiftBlock(initial, gamma2, beta2)

        scaleShiftOut2 = scaleShiftOut2 + initial
        mlpOut = self.pointwiseFeedForward(scaleShiftOut2)

        scaleOut2 = self.scaleBlock(mlpOut, alpha2)        
        updatedZ = scaleShiftOut2 + scaleOut2
        updatedZ = updatedZ[:, xLen + yLen:]
        return updatedZ


    def forward_learning(self, y, z, sharedParameters):
        gamma1, beta1, alpha1, gamma2, beta2, alpha2 = sharedParameters

        yLen, zLen = y.shape[1], z.shape[1]
        yReshaped = self.yProjection(y)

        concatenatedInput = torch.cat([yReshaped, z], dim=1)
        initial = concatenatedInput

        scaleShiftOut1 = self.scaleShiftBlock(concatenatedInput, gamma1, beta1)
        selfAttentionOut1 = self.multiHeadAttention(scaleShiftOut1)
        scaleOut1 = self.scaleBlock(selfAttentionOut1, alpha1)

        initial =  initial + scaleOut1

        scaleShiftOut2 = self.scaleShiftBlock(initial, gamma2, beta2)

        scaleShiftOut2 = scaleShiftOut2 + initial
        mlpOut = self.pointwiseFeedForward(scaleShiftOut2)

        scaleOut2 = self.scaleBlock(mlpOut, alpha2)        
        updatedY = scaleShiftOut2 + scaleOut2
        updatedY = updatedY[:, :yLen]
        updatedY = self.yrevProjection(updatedY)
        return updatedY

    def forward_outputHead(self, y):
        output = self.outputHead(y)
        return output


trm = TinyRecursiveBlock(768, 16)
x_input = torch.randn(2, 576, 768)
y_out = torch.randn(2, 128, 64)
latent = torch.randn(2, 128, 768)

embedDimension = 768
tEmbed = TimeEmbedding(embedDimension=embedDimension)
tEmbed.to(device)
time = torch.tensor([10]).to(device)
tout = tEmbed(time)
adaNorm = AdaptiveLayerNorm(768)
sharedParameters = adaNorm(tout)

z_ = trm.forward_reasoning(x_input, y_out, latent, sharedParameters)
y_ = trm.forward_learning(y_out, z_, sharedParameters)
print(y_.shape)
out = trm.forward_outputHead(y_)
z_.shape, y_.shape, out.shape

torch.Size([2, 128, 64])


(torch.Size([2, 128, 768]), torch.Size([2, 128, 64]), torch.Size([2, 128, 64]))

In [ ]:
def latentRecursion(trm, x, y, z, sharedParameters, n = 6):
    for i in range(n):
        z = trm.forward_reasoning(x, y, z, sharedParameters)

    print(y.shape, z.shape)
    y = trm.forward_learning(y, z, sharedParameters)
    return y, z


def deepReasoning(trm, x, y, z, sharedParameters, n = 6, T = 3):

    with torch.no_grad():
        for j in range(T-1):
            print(x.shape, y.shape, z.shape, "No Grad Deep Reasoning: ")
            y, z = latentRecursion(trm, x, y, z, sharedParameters, n)
    
    print(x.shape, y.shape, z.shape, "With Grad Deep Reasoning: ")
    y, z = latentRecursion(trm, x, y, z, sharedParameters, n)

    output = trm.forward_outputHead(y)

    return y.detach(), z.detach(), output


embedDimension = 768
tEmbed = TimeEmbedding(embedDimension=embedDimension)
tEmbed.to(device)
time = torch.tensor([10]).to(device)
tout = tEmbed(time)
adaNorm = AdaptiveLayerNorm(768)
sharedParameters = adaNorm(tout)


trm = TinyRecursiveBlock(768, 16)

x = torch.randn(2, 576, 768)
y = torch.randn(2, 128, 64)
z = torch.randn(2, 128, 768)
# print(": -- :", x.shape, y.shape, z.shape)
NSup = 16
for i in range(0, NSup):
    y, z, out = deepReasoning(trm, x, y, z, sharedParameters, n = 6, T = 3)
    print(i)
print(out.shape)

torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) No Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) No Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) With Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
0
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) No Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) No Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768]) With Grad Deep Reasoning: 
torch.Size([2, 128, 64]) torch.Size([2, 128, 768])
1
torch.Size([2, 576, 768]) torch.Size([2, 128, 64]) torch.Size([2, 128, 768